### Dataset and Task Metadata

In [15]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="acquire_valued_shoppers_challenge",
    dataset_year="2014",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/c/acquire-valued-shoppers-challenge",
    download_description="""
We start with the data from Kaggle.

kaggle competitions download -c acquire-valued-shoppers-challenge
mkdir -p local-data-warehouse/acquire_valued_shoppers_challenge && mv acquire-valued-shoppers-challenge.zip local-data-warehouse/acquire_valued_shoppers_challenge/ && cd local-data-warehouse/acquire_valued_shoppers_challenge/ && unzip acquire-valued-shoppers-challenge.zip && rm acquire-valued-shoppers-challenge.zip testHistory.csv.gz sampleSubmission.csv.gz
""",
    # References
    academic_reference_bibtex=r"""@misc{DMDave2014AcquireValuedShoppersChallenge,
  author = {DMDave and Todd B and Will Cukierski},
  title  = {Acquire Valued Shoppers Challenge},
  year   = {2014},
  howpublished = {\url{https://kaggle.com/competitions/acquire-valued-shoppers-challenge}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="DMDave2014AcquireValuedShoppersChallenge",
    licence="Kaggle Competition Rules",
    data_tags=["Non-IID", "Temporal", "Anonymized"],
    curation_comments="""
We follow the preprocessing from TabRed https://github.com/yandex-research/tabred/tree/main/preprocessing#ecom-offers-acquire-valued-shoppers-by-dmdave (which follows a top solution https://github.com/MLWave/kaggle_acquire-valued-shoppers-challenge).

- The preprocessing by TabRed collapses customer groups into single entries per customer. So we go from grouped-temporal data to only temporal data. Future work could look into a version without preprocessing.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="target",
    time_on="offerdate"
)

## Preprocessing

To start the processing for this dataset, first run the `run_large_data_preprocessing.py` script outside of this notebook to get the initial state of the dataset from which we are starting the curation. This is mostly done to be able to handle the large amount of data.

In [28]:
import pandas as pd
import numpy as np

df = pd.read_parquet(dataset_mold.path / "merged_input_data.parquet")
print(df.shape)

# Add binary columns again
binary_df = (
    df.assign(
        **{f"never_bought_{c}": (df[f"has_bought_{c}"] == 0)
           for c in ["company", "category", "brand"]},

        has_bought_brand_company_category=(
            (df["has_bought_brand"] != 0)
            & (df["has_bought_category"] != 0)
            & (df["has_bought_company"] != 0)
        ),

        has_bought_brand_category=(
            (df["has_bought_brand"] != 0)
            & (df["has_bought_category"] != 0)
        ),

        has_bought_brand_company=(
            (df["has_bought_brand"] != 0)
            & (df["has_bought_company"] != 0)
        ),
    )[
        [
            "never_bought_company",
            "never_bought_category",
            "never_bought_brand",
            "has_bought_brand_company_category",
            "has_bought_brand_category",
            "has_bought_brand_company",
        ]
    ]
    .astype(np.float32)
)
df = pd.concat([df, binary_df], axis=1)
del binary_df

cat_columns = [
    "target",
    "never_bought_company",
    "never_bought_category",
    "never_bought_brand",
    "has_bought_brand_company_category",
    "has_bought_brand_category",
    "has_bought_brand_company",
]
df[cat_columns] = df[cat_columns].astype("category")

df["offerdate"] = pd.to_datetime(df["offerdate"], format="%Y-%m-%d")
df = df.drop(columns=["id"])


df = df.drop(columns=[
    # Duplicated column
    "has_bought_company_q_1",
    "has_bought_company_a_1",
    "has_bought_category_1",
    "has_bought_category_q_1",
    "has_bought_category_a_1",
    "has_bought_brand_1",
    "has_bought_brand_q_1",
    "has_bought_brand_a_1",
    # Constant column
    "has_bought_company_1"
])

df = df.reset_index(drop=True)

(160057, 116)


## Data Checks

In [29]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 160,057
Columns: 112

#### Duplicate Report
Total duplicate rows: 23 (0.01% of dataset)
Duplicate rows ignoring target: 30 (0.02% of dataset)
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [30]:
# Sample Rows
df_head

,total_spend,target,offervalue,offerdate,day_of_week,day_of_month,day_of_year,has_bought_company,has_bought_company_q,has_bought_company_a,has_bought_category,has_bought_category_q,has_bought_category_a,has_bought_brand,has_bought_brand_q,has_bought_brand_a,has_bought_company_3,has_bought_company_q_3,has_bought_company_a_3,has_bought_category_3,has_bought_category_q_3,has_bought_category_a_3,has_bought_brand_3,has_bought_brand_q_3,has_bought_brand_a_3,has_bought_company_7,has_bought_company_q_7,has_bought_company_a_7,has_bought_category_7,has_bought_category_q_7,has_bought_category_a_7,has_bought_brand_7,has_bought_brand_q_7,has_bought_brand_a_7,has_bought_company_14,has_bought_company_q_14,has_bought_company_a_14,has_bought_category_14,has_bought_category_q_14,has_bought_category_a_14,has_bought_brand_14,has_bought_brand_q_14,has_bought_brand_a_14,has_bought_company_21,has_bought_company_q_21,has_bought_company_a_21,has_bought_category_21,has_bought_category_q_21,has_bought_category_a_21,has_bought_brand_21,has_bought_brand_q_21,has_bought_brand_a_21,has_bought_company_28,has_bought_company_q_28,has_bought_company_a_28,has_bought_category_28,has_bought_category_q_28,has_bought_category_a_28,has_bought_brand_28,has_bought_brand_q_28,has_bought_brand_a_28,has_bought_company_60,has_bought_company_q_60,has_bought_company_a_60,has_bought_category_60,has_bought_category_q_60,has_bought_category_a_60,has_bought_brand_60,has_bought_brand_q_60,has_bought_brand_a_60,has_bought_company_90,has_bought_company_q_90,has_bought_company_a_90,has_bought_category_90,has_bought_category_q_90,has_bought_category_a_90,has_bought_brand_90,has_bought_brand_q_90,has_bought_brand_a_90,has_bought_company_120,has_bought_company_q_120,has_bought_company_a_120,has_bought_category_120,has_bought_category_q_120,has_bought_category_a_120,has_bought_brand_120,has_bought_brand_q_120,has_bought_brand_a_120,has_bought_company_150,has_bought_company_q_150,has_bought_company_a_150,has_bought_category_150,has_bought_category_q_150,has_bought_category_a_150,has_bought_brand_150,has_bought_brand_q_150,has_bought_brand_a_150,has_bought_company_180,has_bought_company_q_180,has_bought_company_a_180,has_bought_category_180,has_bought_category_q_180,has_bought_category_a_180,has_bought_brand_180,has_bought_brand_q_180,has_bought_brand_a_180,never_bought_company,never_bought_category,never_bought_brand,has_bought_brand_company_category,has_bought_brand_category,has_bought_brand_company
0,932.85,1,1.0,2013-03-01,5,1,60,0,0.0,0.0,1,1.0,0.99,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,738.84,0,1.0,2013-03-01,5,1,60,0,0.0,0.0,1,1.0,1.58,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,3779.94,1,1.0,2013-03-01,5,1,60,0,0.0,0.0,3,3.0,10.19,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,1342.25,1,1.0,2013-03-01,5,1,60,0,0.0,0.0,3,3.0,4.95,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0

In [31]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,target,category,0,0.0,2,"0, 1"
1,never_bought_company,category,0,0.0,2,"0.0, 1.0"
2,never_bought_category,category,0,0.0,2,"0.0, 1.0"
3,never_bought_brand,category,0,0.0,2,"1.0, 0.0"
4,has_bought_brand_company_category,category,0,0.0,2,"0.0, 1.0"
5,has_bought_brand_category,category,0,0.0,2,"0.0, 1.0"
6,has_bought_brand_company,category,0,0.0,2,"0.0, 1.0"
7,offerdate,datetime64[ns],0,0.0,56,"2013-03-25 00:00:00, 2013-03-26 00:00:00, 2013-04-24 00:00:00, 2013-03-27 00:00:00, 2013-04-23 00:00:00, 2013-04-25 00:00:00, 2013-04-01 00:00:00, 2013-04-26 00:00:00, 2013-03-30 00:00:00, 2013-04-22 00:00:00"
8,total_spend,float64,0,0.0,147140,"712.46, 5336.51, 1372.02, 2595.74, 2253.76, 1135.9, 783.61, 4347.78, 6058.08, 1817.89"
9,offervalue,float64,0,0.0,6,"1.0, 0.75, 2.0, 1.5, 1.25, 3.0"


In [32]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
total_spend,160057.0,5673.646505,172110.106467,-5953.67,48321663.59
offervalue,160057.0,1.255323,0.524645,0.75,3.00
day_of_week,160057.0,3.620541,1.937209,1.00,7.00
day_of_month,160057.0,19.004467,9.632905,1.00,31.00
day_of_year,160057.0,96.164966,15.090164,60.00,120.00
has_bought_company,160057.0,3.500266,31.999006,0.00,9562.00
has_bought_company_q,160057.0,6.204740,108.014206,-3.00,29585.00
has_bought_company_a,160057.0,15.003597,278.383246,-28.47,90477.57
has_bought_category,160057.0,5.715027,33.779835,0.00,6244.00
has_bought_category_q,160057.0,8.628664,106.847771,-1.00,33992.00


In [33]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column                            rank                                    
has_bought_brand_category         1                     0.0  123196  76.97
                                  2                     1.0   36861  23.03
has_bought_brand_company          1                     0.0  102210  63.86
                                  2                     1.0   57847  36.14
has_bought_brand_company_category 1                     0.0  126984  79.34
                                  2                     1.0   33073  20.66
never_bought_brand                1                     1.0   96906  60.54
                                  2                     0.0   63151  39.46
never_bought_category             1                     0.0   87541  54.69
                                  2                     1.0   72516  45.31
never_bought_company              1                     0.0   86741  54.19
                                  2                     1.0   73316  45.81
offerdate                         1     2013-03-25 00:00:00   10922   6.82
                                  2     2013-03-26 00:00:00    9425   5.89
                                  3     2013-04-24 00:00:00    9044   5.65
                                  4     2013-03-27 00:00:00    7896   4.93
                                  5     2013-04-23 00:00:00    7554   4.72
target                            1                       0  116619  72.86
                                  2                       1   43438  27.14

In [34]:
# Target Distribution
target_df

,count,pct
target,,
0,116619,72.86
1,43438,27.14


## Task Curation

In [43]:
from data_foundry.schema import PredictiveMLSplitsMetadata

splits = {}

test_days = 5 # from TabRed
n_splits = 5 # to keep a reasonable amount of train data still
date_col = task_mold.time_on

df = df.sort_values(date_col).copy()
max_date = df[date_col].max()
print("Newest date:", max_date)

for i in range(n_splits):
    test_start = max_date - pd.Timedelta(days=test_days - 1)

    test_mask = (df[date_col] >= test_start) & (df[date_col] <= max_date)
    train_mask = df[date_col] < test_start

    train_ind = df.loc[train_mask].index.tolist()
    test_ind = df.loc[test_mask].index.tolist()

    # move window backwards
    max_date = test_start - pd.Timedelta(days=1)

    splits[i] = {0: (train_ind, test_ind)}
    print(f"\n---- Split {i}:")
    print(len(train_ind), len(test_ind))
    # last date of train, first date of test
    print(df.loc[train_mask, date_col].max(), df.loc[test_mask, date_col].min())
    # target ratio
    print(
        df.loc[train_mask, task_mold.target_column_name].value_counts(normalize=True),
        df.loc[test_mask, task_mold.target_column_name].value_counts(normalize=True)
    )


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create splits to simulate a model that is deployed for 5 days before being refit. Thus, we use 5 days in the test in a data split. We create 5 splits going back from the newest date and always take all data before the test period as train data.",
    splits=splits,
)

Newest date: 2013-04-30 00:00:00

---- Split 0:
140701 19356
2013-04-25 00:00:00 2013-04-26 00:00:00
target
0    0.740926
1    0.259074
Name: proportion, dtype: float64 target
0    0.639078
1    0.360922
Name: proportion, dtype: float64

---- Split 1:
110396 30305
2013-04-20 00:00:00 2013-04-21 00:00:00
target
0    0.769865
1    0.230135
Name: proportion, dtype: float64 target
0    0.635506
1    0.364494
Name: proportion, dtype: float64

---- Split 2:
105052 5344
2013-04-15 00:00:00 2013-04-16 00:00:00
target
0    0.769105
1    0.230895
Name: proportion, dtype: float64 target
0    0.784805
1    0.215195
Name: proportion, dtype: float64

---- Split 3:
102294 2758
2013-04-10 00:00:00 2013-04-11 00:00:00
target
0    0.770886
1    0.229114
Name: proportion, dtype: float64 target
0    0.703046
1    0.296954
Name: proportion, dtype: float64

---- Split 4:
90387 11907
2013-04-05 00:00:00 2013-04-06 00:00:00
target
0    0.776738
1    0.223262
Name: proportion, dtype: float64 target
0    0.7264

## Export

In [44]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c6611-15df-7536-8699-e24c3acb32d8
75b422375132dbc45da3521c87598e21d7a02721232a5acb274ecef10d26d180
